# Emory single-entity visualization

Visualize the canonical demo entity `1827183_359559206` (Emory sepsis case, record `B035-0564111269`).

Verifies alignment between:
1. **WAV** — PLETH40 + II120 (derived from raw WFDB)
2. **VITAL** — `_0n.mat` dense numerics at 0.5 Hz (HR, SPO2-%, RESP)
3. **EHR** — JGSEPSIS_VITALS2 + JGSEPSIS_LABS (NY local → UTC)
4. **LIST CSV** — whole-list `wfdb_start`, task-list `valid_start/end`, `sepsis_time_zero_dttm`

**Run environment**: `/labs/hulab/mxwang/anaconda3/envs/physio_data/bin/python` on `bedanalysis`.

Change `ZOOM_START_MIN` and `ZOOM_SPAN_MIN` in the zoom cells to inspect different windows.

In [ ]:
import os, json
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from dateutil.relativedelta import relativedelta

import numpy as np
import polars as pl
import wfdb
from scipy.io import loadmat
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

UTC = timezone.utc
NY = ZoneInfo('America/New_York')

# --- paths ---
ENC = 359559206
EMPI = 1827183
REC = 'B035-0564111269'
WFDB_ROOT = '/labs/collab/Waveform_Data/Waveform_Data'
CSV_DIR   = '/labs/hulab/Emory_EHR/CDW_Pull_ICU_Data_Siva_version'
ENTITY_DIR = f'/labs/hulab/mxwang/Physio_Data/workzone/emory/explore/demo_entity/{EMPI}_{ENC}'
WHOLE_CSV = '/labs/hulab/mxwang/data/sepsis/Wav/sepsis_cc_2025_06_13_all_collab.csv'
TASK_CSV  = '/labs/hulab/mxwang/data/sepsis/Wav/sepsis_cc_2025_06_13_all_collab_uniq_combine.csv'

print('Entity dir:', ENTITY_DIR)

## 1. Load canonical entity (waveform segments + EHR events + meta)

In [ ]:
with open(f'{ENTITY_DIR}/meta.json') as f:
    meta = json.load(f)

pleth40  = np.load(f'{ENTITY_DIR}/PLETH40.npy', mmap_mode='r')
ii120    = np.load(f'{ENTITY_DIR}/II120.npy',   mmap_mode='r')
time_ms  = np.load(f'{ENTITY_DIR}/time_ms.npy', mmap_mode='r')
events   = np.load(f'{ENTITY_DIR}/ehr_events.npy', mmap_mode='r')

print(f'PLETH40.shape = {pleth40.shape}  II120.shape = {ii120.shape}')
print(f'n_segments = {len(time_ms)}  wave_start = {datetime.fromtimestamp(int(time_ms[0])/1000, UTC)}')
print(f'wave_end   = {datetime.fromtimestamp(int(time_ms[-1])/1000, UTC)}  (last segment start)')
print(f'events = {len(events)}  (baseline={meta["n_baseline"]}, recent={meta["n_recent"]}, future={meta["n_future"]})')
print(f'vars in events = {sorted(set(events["var_id"].tolist()))}')
print(f'meta keys = {list(meta.keys())}')

## 2. Load raw sources for overlay — `_0n.mat` continuous vitals + list CSVs

In [ ]:
# _0n dense vitals (full 25-day record — we'll clip to wave window for display)
rec_dir = f'{WFDB_ROOT}/{REC.split("-")[0]}/{REC}'
nhdr = wfdb.rdheader(f'{rec_dir}/{REC}_0n')
mat = loadmat(f'{rec_dir}/{REC}_0n.mat')
raw = mat['val'].astype(np.float32)
names_0n = list(nhdr.sig_name)
gains_0n = np.asarray(nhdr.adc_gain, dtype=np.float32)
fs_0n = float(nhdr.fs)

def chan(name):
    if name not in names_0n: return None
    i = names_0n.index(name); s = raw[i] / gains_0n[i]
    s[s == 0] = np.nan; s[s < 0] = np.nan; return s

# time axis for _0n in UTC ms
wave_base_ms = int(time_ms[0])
step_ms = 1000.0 / fs_0n
t_0n = wave_base_ms + np.arange(raw.shape[1]) * step_ms

hr_0n   = chan('HR')
spo2_0n = chan('SPO2-%')
resp_0n = chan('RESP')
nbp_s   = chan('NBP-S')
nbp_d   = chan('NBP-D')
nbp_m   = chan('NBP-M')

print(f'_0n fs = {fs_0n} Hz, total samples = {raw.shape[1]}')
print(f'_0n channels: {names_0n}')

# List CSVs
task = pl.read_csv(TASK_CSV).filter(pl.col('encounter_nbr') == ENC).row(0, named=True)
whole = pl.read_csv(WHOLE_CSV).filter(pl.col('encounter_nbr') == ENC)

def naive_utc_ms(s):
    return int(datetime.fromisoformat(str(s)).replace(tzinfo=UTC).timestamp() * 1000)

valid_start_ms = naive_utc_ms(task['valid_start'])
valid_end_ms   = naive_utc_ms(task['valid_end'])
sepsis_t0_ms   = naive_utc_ms(task['sepsis_time_zero_dttm'])
wfdb_start_ms  = int(datetime.fromisoformat(str(whole['wfdb_start'].min()).replace('Z', '+00:00')).timestamp() * 1000)
wfdb_end_ms    = int(datetime.fromisoformat(str(whole['wfdb_end'].max()).replace('Z', '+00:00')).timestamp() * 1000)

print()
print('Task type:', task['type'])
print('Valid start:', datetime.fromtimestamp(valid_start_ms/1000, UTC))
print('Valid end:  ', datetime.fromtimestamp(valid_end_ms/1000, UTC))
print('Sepsis t0:  ', datetime.fromtimestamp(sepsis_t0_ms/1000, UTC))
print('Wave full:  ', datetime.fromtimestamp(wfdb_start_ms/1000, UTC), '->', datetime.fromtimestamp(wfdb_end_ms/1000, UTC))

## 3. Helper — plot a zoom window with all four sources overlaid

Call `plot_zoom(zoom_start_min, zoom_span_min)`. Start is minutes after `wave_start`.

In [ ]:
VAR_IDS = meta.get('var_ids_used_in_demo', [])
NAMES = {
    0: 'Potassium (lab)', 1: 'Calcium (lab)', 2: 'Sodium (lab)', 3: 'Glucose (lab)',
    4: 'Lactate (lab)', 5: 'Creatinine (lab)', 7: 'Platelet (lab)', 9: 'Hemoglobin (lab)',
    100: 'PULSE (EHR)', 101: 'SPO2 (EHR)', 102: 'RR (EHR)', 103: 'TEMP (EHR)',
    104: 'SBP_CUFF (EHR)', 105: 'DBP_CUFF (EHR)', 106: 'MAP_CUFF (EHR)',
    150: 'HR monitor (_0n)', 151: 'SPO2% monitor (_0n)', 152: 'RESP monitor (_0n)',
}
COLORS = {
    100:'tab:red', 101:'tab:orange', 102:'tab:purple', 103:'magenta',
    104:'tab:blue', 105:'tab:cyan', 106:'tab:olive',
    150:'tab:red', 151:'tab:orange', 152:'tab:purple',
    0:'k', 1:'k', 2:'k', 3:'tab:green', 4:'tab:brown', 5:'tab:pink', 7:'tab:gray', 9:'navy',
}

PLETH_FS = meta['channels']['PLETH40']['sample_rate_hz']
II_FS    = meta['channels']['II120']['sample_rate_hz']
SEG_SEC  = meta['segment_duration_sec']

def plot_zoom(zoom_start_min=20, zoom_span_min=30):
    zoom_start_ms = wave_base_ms + int(zoom_start_min * 60_000)
    zoom_end_ms   = zoom_start_ms + int(zoom_span_min * 60_000)
    seg_lo = max(0, (zoom_start_ms - wave_base_ms) // (SEG_SEC * 1000))
    seg_hi = min(len(time_ms), (zoom_end_ms   - wave_base_ms) // (SEG_SEC * 1000) + 1)
    pleth_slice = np.asarray(pleth40[seg_lo:seg_hi]).reshape(-1)
    ii_slice    = np.asarray(ii120[seg_lo:seg_hi]).reshape(-1)
    pleth_t = np.array(wave_base_ms + seg_lo*SEG_SEC*1000, dtype='datetime64[ms]') + np.arange(len(pleth_slice)) * np.timedelta64(int(1000/PLETH_FS), 'ms')
    ii_t    = np.array(wave_base_ms + seg_lo*SEG_SEC*1000, dtype='datetime64[ms]') + np.arange(len(ii_slice))    * np.timedelta64(int(1000/II_FS),    'ms')
    
    # _0n clip
    m0n = (t_0n >= zoom_start_ms) & (t_0n <= zoom_end_ms)
    t0n_dt = np.array(t_0n[m0n], dtype='datetime64[ms]')
    
    # event clip
    mev = (events['time_ms'] >= zoom_start_ms) & (events['time_ms'] <= zoom_end_ms)
    ev_zoom = events[mev]
    
    fig, axes = plt.subplots(5, 1, figsize=(15, 13), sharex=True)
    axes[0].plot(pleth_t, pleth_slice, lw=0.5, color='steelblue'); axes[0].set_ylabel('PLETH40'); axes[0].grid(alpha=0.3)
    axes[0].set_title(f'Encounter {EMPI}_{ENC} — {zoom_start_min} min to {zoom_start_min+zoom_span_min} min (UTC)')
    axes[1].plot(ii_t, ii_slice, lw=0.4, color='darkgreen'); axes[1].set_ylabel('II120 (mV)'); axes[1].grid(alpha=0.3)
    
    ax = axes[2]
    if hr_0n is not None:   ax.plot(t0n_dt, hr_0n[m0n],   lw=0.6, color='tab:red',    label='HR')
    if spo2_0n is not None: ax.plot(t0n_dt, spo2_0n[m0n], lw=0.6, color='tab:orange', label='SPO2%')
    if resp_0n is not None: ax.plot(t0n_dt, resp_0n[m0n], lw=0.6, color='tab:purple', label='RESP')
    ax.set_ylabel('_0n\ncontinuous'); ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
    
    ax = axes[3]
    if nbp_s is not None: ax.plot(t0n_dt, nbp_s[m0n], lw=0.6, color='tab:blue',  label='NBP-S (_0n hold)')
    if nbp_m is not None: ax.plot(t0n_dt, nbp_m[m0n], lw=0.6, color='tab:olive', label='NBP-M (_0n hold)')
    if nbp_d is not None: ax.plot(t0n_dt, nbp_d[m0n], lw=0.6, color='tab:cyan',  label='NBP-D (_0n hold)')
    for vid, marker, color in [(104, 'x', 'blue'), (105, 'x', 'cyan'), (106, 'x', 'olive')]:
        m = ev_zoom['var_id'] == vid
        if m.any():
            t = np.array(ev_zoom['time_ms'][m], dtype='datetime64[ms]')
            ax.scatter(t, ev_zoom['value'][m], s=50, color=color, marker=marker, label=f'EHR {NAMES[vid]}')
    ax.set_ylabel('BP\n(_0n hold vs EHR event)'); ax.legend(fontsize=7, loc='upper right', ncol=2); ax.grid(alpha=0.3)
    
    ax = axes[4]
    for vid in [100, 101, 102, 103] + [0, 1, 2, 3, 4, 5, 7, 9]:
        m = ev_zoom['var_id'] == vid
        if m.any():
            t = np.array(ev_zoom['time_ms'][m], dtype='datetime64[ms]')
            c = COLORS.get(vid, 'k')
            ax.scatter(t, ev_zoom['value'][m], s=40, color=c, marker='x', label=NAMES.get(vid, f'var{vid}'))
    ax.set_ylabel('EHR events\n(vitals + labs)'); ax.set_xlabel('UTC time'); ax.legend(fontsize=7, loc='upper right', ncol=3); ax.grid(alpha=0.3)
    
    loc = mdates.AutoDateLocator(minticks=6, maxticks=12)
    for ax in axes:
        ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    fig.autofmt_xdate(rotation=30); fig.tight_layout(); plt.show()

plot_zoom(zoom_start_min=20, zoom_span_min=30)

## 4. Zoom to a single cuff event (2-minute window)

This is the tightest alignment check — EHR `SBP_CUFF` event time vs `_0n` `NBP-S` hold-signal step.

In [ ]:
# Find first cuff event in wave window
cuff_events = events[events['var_id'] == 104]
if len(cuff_events) > 0:
    t0 = int(cuff_events['time_ms'][0])
    zoom_start = (t0 - wave_base_ms) // 60_000 - 1   # 1 min before
    plot_zoom(zoom_start_min=int(zoom_start), zoom_span_min=2)
    print(f'First cuff event: t={datetime.fromtimestamp(t0/1000, UTC)}  SBP_CUFF={cuff_events["value"][0]}')
else:
    print('No SBP_CUFF events in entity')

## 5. Full 8-hour overview (all events + `_0n` trend)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)

# Panel 1: _0n HR + SPO2% + RESP (full 8h)
m0n_all = (t_0n >= wave_base_ms) & (t_0n <= int(time_ms[-1]) + SEG_SEC*1000)
t0n_dt = np.array(t_0n[m0n_all], dtype='datetime64[ms]')
ax = axes[0]
if hr_0n is not None:   ax.plot(t0n_dt, hr_0n[m0n_all],   lw=0.5, color='tab:red',    label='HR')
if spo2_0n is not None: ax.plot(t0n_dt, spo2_0n[m0n_all], lw=0.5, color='tab:orange', label='SPO2%')
if resp_0n is not None: ax.plot(t0n_dt, resp_0n[m0n_all], lw=0.5, color='tab:purple', label='RESP')
ax.set_ylabel('_0n continuous'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title(f'Full 8-hour wave window for {EMPI}_{ENC}  ({task["type"]}, sepsis_t0 in {(sepsis_t0_ms-wave_base_ms)/86400000:.1f} d)')

# Panel 2: BP (_0n hold + EHR cuff overlay)
ax = axes[1]
if nbp_s is not None: ax.plot(t0n_dt, nbp_s[m0n_all], lw=0.5, color='tab:blue',  alpha=0.4, label='NBP-S (_0n hold)')
if nbp_m is not None: ax.plot(t0n_dt, nbp_m[m0n_all], lw=0.5, color='tab:olive', alpha=0.4, label='NBP-M (_0n hold)')
if nbp_d is not None: ax.plot(t0n_dt, nbp_d[m0n_all], lw=0.5, color='tab:cyan',  alpha=0.4, label='NBP-D (_0n hold)')
for vid, marker, color in [(104, 'x', 'blue'), (105, 'x', 'cyan'), (106, 'x', 'olive')]:
    m = events['var_id'] == vid
    if m.any():
        t = np.array(events['time_ms'][m], dtype='datetime64[ms]')
        ax.scatter(t, events['value'][m], s=20, color=color, marker=marker, label=NAMES[vid], zorder=3)
ax.set_ylabel('BP'); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

# Panel 3: EHR + labs events
ax = axes[2]
for vid in [100, 101, 102, 103] + [0, 1, 2, 3, 4, 5, 7, 9]:
    m = events['var_id'] == vid
    if m.any():
        t = np.array(events['time_ms'][m], dtype='datetime64[ms]')
        c = COLORS.get(vid, 'k')
        ax.scatter(t, events['value'][m], s=25, color=c, marker='o' if vid < 100 else 'x', label=NAMES.get(vid, f'v{vid}'))
ax.set_ylabel('EHR events'); ax.set_xlabel('UTC time'); ax.legend(fontsize=7, ncol=3, loc='upper right'); ax.grid(alpha=0.3)

loc = mdates.AutoDateLocator(minticks=6, maxticks=10)
for ax in axes:
    ax.xaxis.set_major_locator(loc); ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 6. Summary — time alignment table

All four time sources, converted to UTC ms, for this encounter:

In [ ]:
import pandas as pd
rows = [
    ('LIST CSV — whole wfdb_start (Z)', wfdb_start_ms),
    ('LIST CSV — whole wfdb_end (Z)',   wfdb_end_ms),
    ('LIST CSV — task valid_start (naive→UTC)', valid_start_ms),
    ('LIST CSV — task valid_end (naive→UTC)',   valid_end_ms),
    ('LIST CSV — sepsis_time_zero (naive→UTC)', sepsis_t0_ms),
    ('WAV — _0000 first segment start',         int(time_ms[0])),
    ('WAV — last 30s segment start',            int(time_ms[-1])),
    ('VITAL — _0n first sample',                int(wave_base_ms)),
    ('VITAL — _0n last sample',                 int(wave_base_ms + (raw.shape[1]-1)*step_ms)),
    ('EHR — first event (all vars)',            int(events["time_ms"].min())),
    ('EHR — last event',                        int(events["time_ms"].max())),
]
df = pd.DataFrame(rows, columns=['source', 'utc_ms'])
df['utc_iso'] = df['utc_ms'].apply(lambda x: datetime.fromtimestamp(x/1000, UTC).isoformat())
df['rel_to_wave_start_min'] = ((df['utc_ms'] - wave_base_ms) / 60_000).round(2)
df